In [27]:
import json, pandas as pd
import os, requests, json, time
from pathlib import Path
import numpy as np
API_KEY = "oe_3ZkPHh9ctU2nmLzHQhVi26tZ"  

In [9]:
r = requests.get(
    "https://api.openelectricity.org.au/v4/me",
    headers={"Authorization": f"Bearer {API_KEY}", "Accept": "application/json"},
    timeout=20
)
print("status:", r.status_code)
print(r.text)
r.raise_for_status()


status: 200
{"data":{"id":"key_4zUjtQoqatD3GUup","full_name":"Shixing Xu","email":"xushixing2023@gmail.com","owner_id":"user_34RR6SIJEYDgJVE6Cbq7RNEkDzm","plan":"BASIC","meta":{"remaining":495,"reset":"2025-10-29T23:36:25.902000Z"}}}


In [19]:
API_BASE = "https://api.openelectricity.org.au/v4"
API_KEY = "oe_3ZkPHh9ctU2nmLzHQhVi26tZ"
HEADERS = {"Authorization": f"Bearer {API_KEY}"}

url = f"{API_BASE}/data/facilities/NEM"
common = {
    "interval": "5m",
    "facility_code": ["ERARING"],              
    "date_start": "2025-10-06T00:00:00",     
    "date_end":   "2025-10-12T23:55:00",
}

Path("data_raw").mkdir(exist_ok=True)

# 1) power
p = requests.get(url, headers=HEADERS, params={**common, "metrics": ["power"]}, timeout=60)
p.raise_for_status()
with open("data_raw/NEM_ERARING_power_5m.json", "w", encoding="utf-8") as f:
    json.dump(p.json(), f, ensure_ascii=False, indent=2)
print("power saved")

time.sleep(0.5)  

# 2) emissions
e = requests.get(url, headers=HEADERS, params={**common, "metrics": ["emissions"]}, timeout=60)
e.raise_for_status()
with open("data_raw/NEM_ERARING_emissions_5m.json", "w", encoding="utf-8") as f:
    json.dump(e.json(), f, ensure_ascii=False, indent=2)
print("emissions saved")


power saved
emissions saved


In [21]:
#  power.csv

with open("data_raw/NEM_ERARING_power_5m.json", encoding="utf-8") as f:
    j_power = json.load(f)

rows = []
for block in j_power.get("data", []):
    if block.get("metric") != "power":
        continue
    for series in block.get("results", []):
        unit_code = series.get("columns", {}).get("unit_code") or series.get("name")
        for ts, val in series.get("data", []):
            rows.append({"timestamp": ts, "unit_code": unit_code, "power_MW": val})
df_power = pd.DataFrame(rows)
df_power.to_csv("data_raw/NEM_ERARING_power_5m.csv", index=False)
print("saved： data_raw/NEM_ERARING_power_5m.csv")

# emissions.csv
with open("data_raw/NEM_ERARING_emissions_5m.json", encoding="utf-8") as f:
    j_emis = json.load(f)

rows = []
for block in j_emis.get("data", []):
    if block.get("metric") != "emissions":
        continue
    for series in block.get("results", []):
        unit_code = series.get("columns", {}).get("unit_code") or series.get("name")
        for ts, val in series.get("data", []):
            rows.append({"timestamp": ts, "unit_code": unit_code, "emissions_tCO2e": val})
df_emis = pd.DataFrame(rows)
df_emis.to_csv("data_raw/NEM_ERARING_emissions_5m.csv", index=False)
print(" saved ： data_raw/NEM_ERARING_emissions_5m.csv")

saved： data_raw/NEM_ERARING_power_5m.csv
 saved ： data_raw/NEM_ERARING_emissions_5m.csv


In [29]:
# 1.read files
df_p = pd.read_csv("data_raw/NEM_ERARING_power_5m.csv")
df_e = pd.read_csv("data_raw/NEM_ERARING_emissions_5m.csv")
# 2. exchange time
df_p["timestamp"] = pd.to_datetime(df_p["timestamp"])
df_e["timestamp"] = pd.to_datetime(df_e["timestamp"])
# 3. merge
df = pd.merge(df_p, df_e, on=["timestamp", "unit_code"], how="outer").sort_values("timestamp")
# 4.data clean
df["power_MW"] = pd.to_numeric(df["power_MW"], errors="coerce")
df["emissions_tCO2e"] = pd.to_numeric(df["emissions_tCO2e"], errors="coerce") #type transfer
df = df.drop_duplicates(subset=["timestamp", "unit_code"]) #delete multiply
df = df[(df["power_MW"] >= 0) & (df["emissions_tCO2e"] >= 0)] #delete outlier
df[["power_MW", "emissions_tCO2e"]] = df[["power_MW", "emissions_tCO2e"]].replace(0, np.nan)
# 5.emission intensity
df["emission_intensity_tCO2_MWh"] = df["emissions_tCO2e"] / df["power_MW"]
# 6. 导出
df.to_csv("data_raw/NEM_ERARING_power_emissions_cleaned.csv", index=False)
print("saved → data_raw/NEM_ERARING_power_emissions_cleaned.csv")



saved → data_raw/NEM_ERARING_power_emissions_cleaned.csv
